In [11]:
!pip install --upgrade transformers

In [12]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset

In [13]:
# Load the dataset containing customer support tickets from Kaggle input path
# This dataset includes ticket text and their corresponding category labels
df = pd.read_csv("/kaggle/input/datasets/warcoder/customer-support-ticket-tagging/customer_tickets.csv")

# Rename columns for consistency and easier access in the pipeline
# 'text' → input ticket content
# 'labels' → target category for classification
df.columns = ["text","labels"]
df.head()

,text,labels
0,"Dear Customer Support Team, We are experiencin...",Technical Support
1,"Dear Customer Support,<br><br>I hope this mess...",Product Support
2,"Dear Tech Online Store Customer Support,\n\nI ...",Returns and Exchanges
3,"Dear IT Services Customer Support, \n\nWe are ...",Product Support
4,"Greetings IT Services Customer Support,\n\nI a...",Technical Support


In [14]:
# Remove any rows that contain missing values (NaN) in either text or labels columns
# This is important because missing data can break model training or cause errors during tokenization
df.dropna(inplace=True)

In [15]:
# Initialize LabelEncoder to convert categorical text labels into numeric format
# Machine learning models cannot understand text labels directly, so we encode them as numbers
label_encoder = LabelEncoder()

# Convert string labels (e.g., Billing, Technical, IT Support) into numeric values (0, 1, 2, ...)
df['labels'] = label_encoder.fit_transform(df['labels']) 

# Convert Pandas DataFrame into Hugging Face Dataset format for efficient training
# We only keep 'text' (input) and 'labels' (target) columns
dataset = Dataset.from_pandas(df[['text', 'labels']])

# Split dataset into training and testing sets
# 14.5% of data is used for testing, remaining for training
hf_dataset = dataset.train_test_split(test_size = 0.145)

# Print dataset structure to verify successful split
print(hf_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', '__index_level_0__'],
        num_rows: 288
    })
    test: Dataset({
        features: ['text', 'labels', '__index_level_0__'],
        num_rows: 50
    })
})


In [16]:
# Display all unique label categories learned by the LabelEncoder
# This shows the mapping between original text labels and their numeric values
label_encoder.classes_

array(['Billing and Payments', 'Customer Service', 'General Inquiry',
       'Human Resources', 'IT Support', 'Product Support',
       'Returns and Exchanges', 'Sales and Pre-Sales',
       'Service Outages and Maintenance', 'Technical Support'],
      dtype=object)

In [17]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Define the pre-trained transformer model name (DeBERTa v3 base)
# This model already understands language and will be fine-tuned for our classification task
model_name = "microsoft/deberta-v3-base"  

# Load the tokenizer corresponding to the model
# Tokenizer converts raw text into numerical tokens that the model can process
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load pre-trained DeBERTa model and adapt it for text classification
# num_labels defines how many output categories (classes) the model should predict
# This modifies the final layer of the model for our specific classification task
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(label_encoder.classes_))

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier

In [18]:
# Function to preprocess raw text data into tokenized format
# This is required because the transformer model cannot work directly with text
def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, padding=True)

# Apply the preprocessing function to the entire dataset in batches
# batched=True improves speed by processing multiple examples at once
tokenized_datasets = hf_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/288 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [19]:
# Define training configuration (hyperparameters and settings for model training)
training_args = TrainingArguments(
    output_dir="./results",              # Directory where model checkpoints and outputs will be saved
    learning_rate=0.00002,               # Step size for updating model weights (how fast the model learns)
    per_device_train_batch_size=8,       # Number of training samples processed at once
    per_device_eval_batch_size=8,        # Number of evaluation samples processed at once
    num_train_epochs=10,                 # Number of times the model goes through the entire dataset
    weight_decay=0.05,                   # Regularization to reduce overfitting (penalizes large weights)
    logging_dir='./logs',                # Folder to store training logs
    logging_steps=30,                    # Show training progress every 30 steps
    report_to='none'                     # Disable external logging tools (like Weights & Biases)
)

# Trainer handles the complete training pipeline (forward pass, loss, backpropagation, optimization)
trainer = Trainer(
    model=model,                                   # Pre-trained transformer model to be trained
    args=training_args,                            # Training configuration defined above
    train_dataset=tokenized_datasets['train'],     # Training data used to learn patterns
    eval_dataset=tokenized_datasets['test'],       # Test data used to evaluate performance

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [20]:
# Start the training process
# This is where the model actually learns patterns from the training data

# Inside this step, the model will:
# 1. Take input batches from train_dataset
# 2. Perform forward pass (make predictions)
# 3. Compute loss (difference between prediction and actual label)
# 4. Perform backpropagation (calculate gradients)
# 5. Update model weights using optimizer
# 6. Repeat this process for all epochs

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
30,0.000000
60,0.000000
90,0.000000
120,0.000000
150,0.000000
180,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=180, training_loss=0.0, metrics={'train_runtime': 62.2531, 'train_samples_per_second': 46.263, 'train_steps_per_second': 2.891, 'total_flos': 506205326983680.0, 'train_loss': 0.0, 'epoch': 10.0})

In [21]:
# Evaluate the trained model on the test dataset
# This checks how well the model performs on unseen data

# During evaluation:
# - The model makes predictions on test data
# - Attention mechanism is used internally to understand word relationships
# - Attention mask is also used to ignore padding tokens
# - Loss and evaluation metrics (like accuracy) are calculated
# - IMPORTANT: No learning happens here (no weight updates)

trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Step
0.000000,nan,180


{'eval_loss': nan}

In [22]:
#Saves the trained AI model
#This includes:
#learned weights (what model has learned from data)
#classification head (your labels logic)
trainer.save_model("./text-classification-model")

#Saves the tokenizer (text processor)
#It stores:vocabulary (word list)
#rules for splitting text
#special tokens ([PAD], [CLS], etc.)
tokenizer.save_pretrained("./text-classification-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./text-classification-model/tokenizer_config.json',
 './text-classification-model/tokenizer.json')

In [23]:
from transformers import pipeline

# loading the locally saved model
classifier = pipeline("text-classification", model="./text-classification-model", tokenizer=tokenizer)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [24]:
# Predictor Function to evaluate some tickets
# This function takes a support ticket, uses the trained model to predict its category, and converts both original and predicted labels back into human-readable form for comparison.
def predictor(input_ticket,org_label):
    print(f"Input Ticket: {input_ticket}")
    result = classifier(input_ticket)
    print("\n")
    org_label_decoded = label_encoder.inverse_transform([int(org_label)])[0]
    decoded_label = label_encoder.inverse_transform([int(result[0]['label'].split("_")[-1])])[0]
    print("Original Label: ",org_label,",Original Label Decoded: ",org_label_decoded)
    print(f"Predicted Label: {int(result[0]['label'].split('_')[-1])} ,Predicted label Decoded: {decoded_label}")

In [25]:
predictor(df['text'][319],df['labels'][319])

Input Ticket: I am unable to connect to the Wi-Fi.


Original Label:  1 ,Original Label Decoded:  Customer Service
Predicted Label: 0 ,Predicted label Decoded: Billing and Payments


In [26]:
predictor(df['text'][338],df['labels'][338])

Input Ticket: Dear Customer Support Team,

I am contacting you to seek prompt professional help regarding our IT Consulting Service. We are facing an urgent requirement for server setup and network enhancement. Our systems are presently experiencing difficulties that may negatively affect our business activities. It is imperative that we address these issues swiftly to avoid any interruptions.

Could you kindly prioritize our request and allocate an expert to help us with these concerns? We need someone with specialized expertise in server setups and optimization methods. Please inform us at your earliest convenience about the availability of your support personnel.

We are ready for a consultation call whenever it suits you to provide any additional information needed. You can reach me at <tel_num>.

Thank you for your prompt attention to this issue. We anticipate your swift reply.

Best regards,

<name>


Original Label:  9 ,Original Label Decoded:  Technical Support
Predicted Label: